# 3.1 SIMT算子开发

本节介绍SIMT(单指令多线程)算子开发流程，对各步骤要点进行讲解。
- 理解simt编程模型
- 开发多核并行算子

本章将以 ScatterUpdate 算子为例，详细讲解 Simt算子的完整开发流程，涵盖以下内容：
- simt概述
- 环境准备
- 算子分析（业务背景、Tiling策略）
- 工程结构与Tiling数据设计
- Host侧代码实现详解
- Kernel侧代码实现详解
- 编译与运行
- 测试与验证
- 课后实践
---

## **1. SIMT概述**
在SIMT编程中Global Memory上的数据可以被直接读取和使用。SIMT编程常通过组织线程的层次结构来实现数据切分，使用threadIdx等SIMT BuiltIn关键字计算线程应处理的数据索引，完成对应数据的计算，从而将函数的实现简化为标量计算。
SIMT是Ascend C单指令多线程的编程抽象，允许一条指令对数据进行独立寻址，更加灵活， 如下图所示，对于离散访问的算子，适合使用SIMT编程实现，例如Scatter类与scatter_update类算子。此外，线程可以独立执行，每个线程具有较高的灵活性，能够执行不同的逻辑分支以完成复杂的逻辑实现。  </br>
<img src="./images/simt_introduction.png" alt="simt_introduction_flow"  width="700px" >


---
## **2. 环境准备**
> **运行环境说明**：本章节内容基于 **Ascend 950** 平台开发与验证，采用 **`<<<>>>` Kernel 直调（Direct Invoke）** 方式启动算子，不使用 msopgen 工具或 aclnn 接口。请确保运行环境为 950 系列硬件。

本文所有内容均存放于Sources文件夹。在开始之前，先对jupyter环境进行初始化，导入CANN环境变量并创建代码目录。

In [ ]:
!mkdir -p Sources/03.01

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))

由于本项目是 Kernel 直调工程（不使用 msopgen 生成），需要手动创建目录结构：

In [ ]:
# 创建工程目录结构
!mkdir -p Sources/03.01/scatter_update/op_host/scripts
!mkdir -p Sources/03.01/scatter_update/op_kernel
!touch Sources/03.01/scatter_update/op_host/scatter_update_host.asc
!touch Sources/03.01/scatter_update/op_kernel/scatter_update_tiling_data.h
!touch Sources/03.01/scatter_update/op_kernel/scatter_update_kernel.h
!touch Sources/03.01/scatter_update/run_multi_data.sh
!touch Sources/03.01/scatter_update/build.sh
!touch Sources/03.01/scatter_update/CMakeLists.txt

---
## **2. 算子分析**
### **2.1 算子背景**
scatter_update算子的功能为更新指定索引行位置的张量数据。即将 updates 张量中的值按索引 indices 逐个更新到 var 张量的第 0 维对应位置。更新的行索引由输入indices指定，更新的值由updates张量指定。第i个索引更新算子输出var的计算公式为：
```
var[indices[i]] = updates[i]
```

所以，scatter_update算子有三个输入：var、indices与updates；一个输出：var。
本样例中算子输入var的数据类型支持float、half, indices的数据类型为int32_t，updates的数据类型以及算子输出的数据类型与输入的数据类型相同。

scatter_update支持范围定义如下：
| 输入/输出向量 | 支持数据类型 | 执行核 |
|------|------|--------|
| var | "float16", "float" | AIV |
| indices | "int32" | AIV |
| updates | "float16", "float" | AIV |
| var(输出) | "float16", "float" | AIV |

算子实现思路：
1、计算待更新数据的总个数totalLength
2、根据待更新的总个数进行分核和分线程。每个线程处理一个数据，并且区分尾核处理数据量，以确保尾部线程不会进行无效操作。
在算子实现中，无需使用大量临时变量，为了提高性能，可以直接默认最大线程数（2048）,具体使用的时候可以在此基础上适当减小核函数的最大线程数。

### **2.2 Split Tiling 策略**
本样例以均匀分块方案介绍分块参数的计算，根据输入updates的shape大小totalLength、总核数blockDim计算每个核处理的数据量，假设3个输入的shape分别为(m, n),(k),(k, n), 得到如下参数：
| 参数 | 含义 | 计算方式（M=1024, rankSize=4, minSplitM=128） |
|------|------|------|
| `totalLength` | 需要更新的数量 | `k * n` |
| `lengthPerBlock` | 每个核更新的数据量 | `(totalLength + blockDim - 1) / blockDim` |
| `usedCoreNums` | 实际需要的核数 | `(totalLength + lengthPerBlock - 1) / lengthPerBlock` |
| `tailBlockLength` | 尾核更新的数据量 | `totalLength - (usedCoreNums - 1) * lengthPerBlock` |

---
## **3. 工程结构与Tiling数据设计**

### **3.1 工程目录结构**

本项目采用 Kernel 直调（Direct Invoke）方式，工程目录结构如下：

In [ ]:
# 查看工程目录结构
!tree Sources/03.01/scatter_update -L 2

核心文件职责：

```
scatter_update/
├── CMakeLists.txt                          # CMake构建配置
├── build.sh                                # 一键编译脚本
├── run_multi_data.sh                       # 自动化测试脚本
├── op_host/
│   ├── scatter_update_host.asc  # Host侧主程序
│   └── scripts/ (gen_data.py, verify_result.py)
└── op_kernel/
    |—— scatter_update_kernel.h
    └── scatter_update_tiling_data.h
```

### **3.2 Tiling数据结构定义**

首先定义 Tiling 结构体，它们负责在 Host 和 Device 之间传递算子的维度信息和切分策略。

In [ ]:
%%writefile Sources/03.01/scatter_update/op_kernel/scatter_update_tiling_data.h
#ifndef SCATTER_UPDATE_TILING_DATA_H
#define SCATTER_UPDATE_TILING_DATA_H

#include <cstdint>

struct ScatterUpdateTilingData {
    uint64_t varShape[2];
    uint64_t lengthPerBlock;
    uint64_t tailBlockLength;
};

#endif

In [ ]:
# 查看 Tiling 数据结构定义
!cat Sources/03.01/scatter_update/op_kernel/tiling/scatter_update_tiling_data.h

## **4. kernel实现**
### **4.1 Kernel计算模块**

In [ ]:
%%writefile Sources/03.01/scatter_update/op_kernel/scatter_update_kernel.h
#ifndef SCATTER_UPDATE_KERNEL_H
#define SCATTER_UPDATE_KERNEL_H

#include "kernel_operator.h"
#include "simt_api/asc_simt.h"
#include "scatter_update_tiling_data.h"

namespace ScatterUpdateImpl {

using namespace AscendC;
constexpr static uint32_t THREAD_NUM = 2048;
template<typename DataType, typename IdxType>
class ScatterUpdateKernelImpl {
public:
    __aicore__ inline ScatterUpdateKernelImpl() {};
    __aicore__ inline void Init(GM_ADDR var, GM_ADDR indices, GM_ADDR updates, const ScatterUpdateTilingData* tilingData);
    __aicore__ inline void Process();

private:
    uint64_t blockIdx_ = 0;
    uint64_t blockNums_ = 0;
    uint64_t currentLength_ = 0;
    uint64_t start_ = 0;
    uint64_t end_ = 0;
    uint64_t lengthPerBlock_ = 0;
    uint64_t tailBlockLength_ = 0;
    uint64_t totalCol_ = 0;
    uint64_t varFirstDimSize_ = 0;

    GM_ADDR varAddr_;
    GM_ADDR indicesAddr_;
    GM_ADDR updatesAddr_;
};

template<typename DataType, typename IdxType>
__aicore__ inline void ScatterUpdateKernelImpl<DataType, IdxType>::Init(
    GM_ADDR var, GM_ADDR indices, GM_ADDR updates, const ScatterUpdateTilingData* tilingData)
{
    blockIdx_ = GetBlockIdx();
    blockNums_ = GetBlockNum();
    lengthPerBlock_ = tilingData->lengthPerBlock;
    tailBlockLength_ = tilingData->tailBlockLength;
    totalCol_ = tilingData->varShape[1];
    varFirstDimSize_ = tilingData->varShape[0];

    // 计算当前核处理数据的首尾地址
    start_ = blockIdx_ * lengthPerBlock_;
    currentLength_ = (blockIdx_ == blockNums_ - 1) ? tailBlockLength_ : lengthPerBlock_;
    end_ = start_ + currentLength_;
    varAddr_ = var;
    indicesAddr_ = indices;
    updatesAddr_ = updates;
}

template<typename DataType, typename IdxType>
__simt_vf__ __aicore__ LAUNCH_BOUND(THREAD_NUM) inline void ScatterUpdateSimtCompute(uint64_t start, uint64_t end,
    uint64_t totalCol, uint64_t varFirstDimSize, uint64_t varStride, __gm__ DataType* var, __gm__ IdxType* indices, 
    __gm__ DataType* updates)
{
    for (uint64_t curID = start + threadIdx.x; curID < end; curID += blockDim.x) {
        // 计算当前线程ID对应的indices行、var行以及需要更新的位置索引
        uint64_t indiceRow = curID / totalCol;
        uint64_t varRow = static_cast<uint64_t>(indices[indiceRow]);
        uint64_t tailRowIdx = curID - indiceRow * totalCol;             
        uint64_t varDataIdx = varRow * varStride + tailRowIdx;
        
        //越界判断
        if (!(varRow >= 0 && varRow < varFirstDimSize)) {
                continue;
        }

        // 更新
        var[varDataIdx] = updates[curID];
    }
}

template<typename DataType, typename IdxType>
__aicore__ inline void ScatterUpdateKernelImpl<DataType, IdxType>::Process()
{
    asc_vf_call<ScatterUpdateSimtCompute<DataType, IdxType>>(dim3(THREAD_NUM), 
            start_, end_, totalCol_, varFirstDimSize_, totalCol_,
            (__gm__ DataType*)varAddr_, (__gm__ IdxType*)indicesAddr_, (__gm__ DataType*)updatesAddr_);
}

}

#endif

In [ ]:
# 查看 Kernel 核心实现（前80行）
!head -80 Sources/03.01/scatter_update/op_kernel/include/kernel/scatter_update_kernel.h

### **4.2 Kernel入口函数**
算子直调的 Kernel 入口函数：

__global__ __aicore__ void ScatterUpdate(
    __gm__ uint8_t* var, __gm__ uint8_t* indices, __gm__ uint8_t* updates,
    __gm__ uint8_t* workspaceGM, ScatterUpdateTilingData tilingData)
{
    KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_MIX_AIV_1_0);

    ScatterUpdateImpl::ScatterUpdateKernelImpl<float, int32_t> op;
    op.Init(var, indices, updates, &tilingData);
    op.Process();
}

**入口要点：**
- **`KERNEL_TYPE_MIX_AIV_1_0`**：声明 Kernel 为 AIV 混合模式，核按 1:0 比例分区
- **`ScatterUpdate`**：`__global__ __aicore__` Kernel，通过 `<<<>>>` 语法启动, 创建 `ScatterUpdateKernel` 对象


## **5. Host侧实现**
Host 侧负责Tiling 构造和 Kernel 启动。

In [ ]:
%%writefile Sources/03.01/scatter_update/op_host/scatter_update_host.asc
/**
 * @file scatter_update_host.cpp
 * @brief Host-side direct invocation code for ScatterUpdate SIMT kernel
 *
 * ScatterUpdate operation:
 *   var[indices[i], :] = updates[i, :]
 *   - var shape: [VAR_ROW, COL] (float)
 *   - indices shape: [ROW] (int32_t)
 *   - updates shape: [ROW, COL] (float)
 *
 * This program reads input binary files, launches the scatter_update kernel
 * on the NPU via the RunScatterUpdate host wrapper, and writes the updated
 * var to an output binary file.
 */

#include "acl/acl.h"
#include "acl/acl_rt.h"
#include "kernel_operator.h"
#include "simt_api/asc_simt.h"
#include "platform/platform_ascendc.h"
#include <iostream>
#include <fstream>
#include <getopt.h>
#include <vector>
#include <numeric>
#include <random>
#include <limits>
#include "../op_kernel/scatter_update_kernel.h"

// Kernel 入口函数
__global__ __aicore__ void ScatterUpdate(
    __gm__ uint8_t* var, __gm__ uint8_t* indices, __gm__ uint8_t* updates,
    __gm__ uint8_t* workspaceGM, ScatterUpdateTilingData tilingData)
{
    KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_MIX_AIV_1_0);

    ScatterUpdateImpl::ScatterUpdateKernelImpl<float, int32_t> op;
    op.Init(var, indices, updates, &tilingData);
    op.Process();
}

#define ACL_CHECK(call)                                                        \
    do {                                                                       \
        aclError ret = (call);                                                 \
        if (ret != ACL_SUCCESS) {                                              \
            std::cerr << "[ACL ERROR] " << __FILE__ << ":" << __LINE__         \
                      << " code=" << ret << " call=" << #call << std::endl;    \
            return -1;                                                         \
        }                                                                      \
    } while (0)

// ============================================================================
// Helper: write binary file from buffer
// ============================================================================
static bool WriteBinaryFile(const std::string &path, const void *buf,
                            size_t size) {
    std::ofstream ofs(path, std::ios::binary);
    if (!ofs.is_open()) {
        std::cerr << "[ERROR] Cannot create file: " << path << std::endl;
        return false;
    }
    ofs.write(reinterpret_cast<const char *>(buf), size);
    return true;
}

// ============================================================================
// Helper: load entire binary file into a vector
// ============================================================================
static std::vector<uint8_t> LoadBinaryFile(const std::string &path) {
    std::ifstream ifs(path, std::ios::binary | std::ios::ate);
    if (!ifs.is_open()) {
        std::cerr << "[ERROR] Cannot open file: " << path << std::endl;
        return {};
    }
    size_t size = ifs.tellg();
    ifs.seekg(0, std::ios::beg);
    std::vector<uint8_t> data(size);
    ifs.read(reinterpret_cast<char *>(data.data()), size);
    return data;
}

// ============================================================================
// Print usage
// ============================================================================
static void PrintUsage(const char *prog) {
    std::cout << "Usage: " << prog << " [OPTIONS]\n"
              << "Options:\n"
              << "  --m <value>          Number of update rows (ROW)\n"
              << "  --n <value>          Number of var rows (VAR_ROW)\n"
              << "  --k <value>          Number of columns (COL)\n"
              << "  --output_dir <dir>   Working directory for input/output files\n"
              << "  --help               Show this help\n";
}

// ============================================================================
// Main
// ============================================================================
int main(int argc, char **argv) {
    // ---- 解析实际入参的shape信息 ----
    int64_t ROW = 0, VAR_ROW = 0, COL = 0;
    std::string outputDir = ".";

    static struct option longOpts[] = {
        {"m", required_argument, nullptr, 'm'},
        {"n", required_argument, nullptr, 'n'},
        {"k", required_argument, nullptr, 'k'},
        {"output_dir", required_argument, nullptr, 'o'},
        {"help", no_argument, nullptr, 'h'},
        {nullptr, 0, nullptr, 0}};

    int opt;
    while ((opt = getopt_long(argc, argv, "m:n:k:o:h", longOpts, nullptr)) !=
           -1) {
        switch (opt) {
        case 'm':
            ROW = std::atoll(optarg);
            break;
        case 'n':
            VAR_ROW = std::atoll(optarg);
            break;
        case 'k':
            COL = std::atoll(optarg);
            break;
        case 'o':
            outputDir = optarg;
            break;
        case 'h':
        default:
            PrintUsage(argv[0]);
            return 0;
        }
    }

    if (ROW <= 0 || COL <= 0 || VAR_ROW <= 0) {
        std::cerr << "[ERROR] Invalid shape: ROW=" << ROW
                  << " COL=" << COL << " VAR_ROW=" << VAR_ROW << std::endl;
        PrintUsage(argv[0]);
        return -1;
    }

    std::cout << "========== ScatterUpdate Host ==========" << std::endl;
    std::cout << "Parameters: ROW=" << ROW << " COL=" << COL
              << " VAR_ROW=" << VAR_ROW << std::endl;
    std::cout << "Output dir: " << outputDir << std::endl;

    // ---- 输入数据加载 ----
    std::string varFile = outputDir + "/input/var.bin";
    std::string indicesFile = outputDir + "/input/indices.bin";
    std::string updatesFile = outputDir + "/input/updates.bin";
    std::string outputFile = outputDir + "/output/var_updated.bin";

    // var shape: [VAR_ROW, COL], float32
    // indices shape: [ROW], int32
    // updates shape: [ROW, COL], float32
    size_t varSize = VAR_ROW * COL * sizeof(float);
    size_t indicesSize = ROW * sizeof(int32_t);
    size_t updatesSize = ROW * COL * sizeof(float);

    auto varHostData = LoadBinaryFile(varFile);
    auto indicesHostData = LoadBinaryFile(indicesFile);
    auto updatesHostData = LoadBinaryFile(updatesFile);

    if (varHostData.size() != varSize ||
        indicesHostData.size() != indicesSize ||
        updatesHostData.size() != updatesSize) {
        std::cerr << "[ERROR] Input file size mismatch:" << std::endl;
        std::cerr << "  var.bin: expected=" << varSize
                  << " actual=" << varHostData.size() << std::endl;
        std::cerr << "  indices.bin: expected=" << indicesSize
                  << " actual=" << indicesHostData.size() << std::endl;
        std::cerr << "  updates.bin: expected=" << updatesSize
                  << " actual=" << updatesHostData.size() << std::endl;
        return -1;
    }

    // ---- 初始化 AscendCL ----
    ACL_CHECK(aclInit(nullptr));
    int32_t deviceId = 0;
    ACL_CHECK(aclrtSetDevice(deviceId));

    aclrtContext context;
    ACL_CHECK(aclrtCreateContext(&context, deviceId));
    ACL_CHECK(aclrtSetCurrentContext(context));

    aclrtStream stream;
    ACL_CHECK(aclrtCreateStream(&stream));

    std::cout << "AscendCL initialized. Device ID: " << deviceId << std::endl;

    // ---- 分配Device侧内存 ----
    void *devVar = nullptr;
    void *devIndices = nullptr;
    void *devUpdates = nullptr;
    void *devWorkspace = nullptr;

    ACL_CHECK(aclrtMalloc(&devVar, varSize, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(
        aclrtMalloc(&devIndices, indicesSize, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(
        aclrtMalloc(&devUpdates, updatesSize, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&devWorkspace, 64, ACL_MEM_MALLOC_HUGE_FIRST));

    // ---- 将 Host 数据拷贝到 Device侧 ----
    ACL_CHECK(aclrtMemcpy(devVar, varSize, varHostData.data(), varSize,
                          ACL_MEMCPY_HOST_TO_DEVICE));
    ACL_CHECK(aclrtMemcpy(devIndices, indicesSize, indicesHostData.data(),
                          indicesSize, ACL_MEMCPY_HOST_TO_DEVICE));
    ACL_CHECK(aclrtMemcpy(devUpdates, updatesSize, updatesHostData.data(),
                          updatesSize, ACL_MEMCPY_HOST_TO_DEVICE));

    // ---- Tiling 计算 ----
    // totalElements = ROW * COL (total scalar elements in updates)
    uint32_t totalElements = static_cast<uint32_t>(ROW * COL);

    // 获取实际可用物理核数
    auto ascendcPlatform = platform_ascendc::PlatformAscendCManager::GetInstance();
    uint32_t aivCoreNum = ascendcPlatform->GetCoreNumAiv();
    uint32_t usedBlockNum = std::min(aivCoreNum, totalElements);
    if (usedBlockNum == 0) {
        usedBlockNum = 1;
    }
    
    // 每个核按照2048线程计算
    constexpr uint32_t threadsPerBlock = 2048;

    // 每个核实际处理数据量
    uint32_t lengthPerBlock = totalElements / usedBlockNum;
    uint32_t tailBlockLength =
        totalElements - lengthPerBlock * (usedBlockNum - 1);

    // varShape[0] = VAR_ROW (first dim of var), varShape[1] = COL (second dim)
    ScatterUpdateTilingData tilingData;
    tilingData.varShape[0] = static_cast<uint64_t>(VAR_ROW);
    tilingData.varShape[1] = static_cast<uint64_t>(COL);
    tilingData.lengthPerBlock = lengthPerBlock;
    tilingData.tailBlockLength = tailBlockLength;

    std::cout << "Tiling config: usedBlockNum=" << usedBlockNum
              << " threadsPerBlock=" << threadsPerBlock
              << " lengthPerBlock=" << lengthPerBlock
              << " tailBlockLength=" << tailBlockLength << std::endl;
    std::cout << "Var shape: [" << VAR_ROW << ", " << COL << "]" << std::endl;

    // ---- 算子直调方式调用算子kernel ----
    ScatterUpdate<<<usedBlockNum, 0, stream>>>((uint8_t*)devVar, (uint8_t*)devIndices,
                                    (uint8_t*)devUpdates, (uint8_t*)devWorkspace, tilingData);

    ACL_CHECK(aclrtSynchronizeStream(stream));
    std::cout << "Kernel execution completed." << std::endl;

    // ---- 将 Device 侧计算结果拷贝到 HOST ----
    std::vector<uint8_t> resultData(varSize);
    ACL_CHECK(aclrtMemcpy(resultData.data(), varSize, devVar, varSize,
                          ACL_MEMCPY_DEVICE_TO_HOST));

    // ---- 保存输出数据 ----
    if (!WriteBinaryFile(outputFile, resultData.data(), varSize)) {
        std::cerr << "[ERROR] Failed to write output file: " << outputFile
                  << std::endl;
        return -1;
    }
    std::cout << "Output written to: " << outputFile << std::endl;

    // ---- 释放内存等资源 ----
    ACL_CHECK(aclrtFree(devVar));
    ACL_CHECK(aclrtFree(devIndices));
    ACL_CHECK(aclrtFree(devUpdates));
    ACL_CHECK(aclrtFree(devWorkspace));

    ACL_CHECK(aclrtDestroyStream(stream));
    ACL_CHECK(aclrtDestroyContext(context));
    ACL_CHECK(aclrtResetDevice(deviceId));
    ACL_CHECK(aclFinalize());

    std::cout << "========== ScatterUpdate Completed ==========" << std::endl;
    return 0;
}



In [ ]:
# 查看 Host 侧实现（关键函数）
!head -100 Sources/03.01/scatter_update/op_host/scatter_update_host.cpp

**Host 侧关键流程：**

**1. 参数配置（main 函数）：**
- 设置var、indices、updates的shape信息
**2. 设备初始化：**
- `aclInit`：为每个 Device 创建 Context 和 Stream
- `aclrtMalloc` 和 `aclrtMemcpy`：创建Device侧内存资源，并将Host侧数据拷贝到Device侧
**3. Tiling切分：**
- `GetCoreNumAiv`：获取实际可用的物理核数
- 正常核处理数据量 = 总数据量 / 可用核数
- 尾核处理数据量 = 总数据量 - (所用核数 - 1) * 正常核处理数据量
**4. Kernel调度：**
- `ScatterUpdate<<<usedBlockNum, 0, stream>>>(...)`：通过<<<>>>方式设置启动核数，并关联stream。
**5. 同步&数据写回：**
`aclrtSynchronizeStream`：等待Device侧执行结果进行同步
`aclrtMemcpy`：将Device侧数据拷贝到Host侧
**6. 资源释放：**

---
## **6. 构建与运行**

### **6.1 CMakeLists.txt**

In [ ]:
%%writefile Sources/03.01/scatter_update/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

if(NOT DEFINED ENV{ASCEND_HOME_PATH})
    message(FATAL_ERROR "ASCEND_HOME_PATH not set, please configure CANN environment first")
endif()

set(CMAKE_MODULE_PATH "$ENV{ASCEND_HOME_PATH}/x86_64-linux/lib64/cmake" "$ENV{ASCEND_HOME_PATH}/compiler/tikcpp/ascendc_kernel_cmake/asc_modules")
find_package(ASC REQUIRED)

project(ScatterUpdate LANGUAGES C CXX ASC)

set(CMAKE_EXPORT_COMPILE_COMMANDS ON)
set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_POSITION_INDEPENDENT_CODE ON)

if(NOT DEFINED NPU_ARCH)
    set(NPU_ARCH "dav-3510" CACHE STRING "NPU architecture for ASC compiler")
endif()

set(SCATTER_UPDATE_DIR ${CMAKE_CURRENT_SOURCE_DIR})

add_executable(scatter_update
    ${SCATTER_UPDATE_DIR}/op_host/scatter_update_host.asc
)

target_compile_options(scatter_update PRIVATE
    "$<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${NPU_ARCH}>"
    "$<$<COMPILE_LANGUAGE:ASC>:-O3>"
)

target_include_directories(scatter_update PRIVATE
    ${SCATTER_UPDATE_DIR}/op_kernel
    ${SCATTER_UPDATE_DIR}/op_host
)

target_link_libraries(scatter_update PRIVATE
    m
    dl
    platform
    tiling_api
    ascendcl
    runtime
    stdc++
)

install(TARGETS scatter_update
    RUNTIME DESTINATION .
)

install(DIRECTORY
    ${CMAKE_CURRENT_SOURCE_DIR}/op_host/scripts
    DESTINATION .
)

install(FILES
    ${CMAKE_CURRENT_SOURCE_DIR}/run_multi_data.sh
    DESTINATION .
)

**CMake 配置要点：**
- `add_executable` 编译 Host 为可执行文件，包含<<<>>>调度代码
- 头文件引用：`op_kernel`、`op_host`

### **6.2 编译脚本**

In [ ]:
%%writefile Sources/03.01/scatter_update/build.sh
#!/bin/bash

SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
BUILD_DIR="${SCRIPT_DIR}/build"

if [ -z "$ASCEND_HOME_PATH" ]; then
    echo "ERROR: CANN environment not found. Please set ASCEND_HOME_PATH or source set_env.sh"
    exit 1
fi

echo "========== Building ScatterUpdate =========="
echo "Build directory: ${BUILD_DIR}"

rm -rf "${BUILD_DIR}"
mkdir -p "${BUILD_DIR}"
cd "${BUILD_DIR}"

cmake -DCMAKE_INSTALL_PREFIX="${BUILD_DIR}" ..
if [ $? -ne 0 ]; then
    echo "ERROR: cmake failed"
    exit 1
fi

make -j8 install
if [ $? -ne 0 ]; then
    echo "ERROR: make failed"
    exit 1
fi

echo "========== Build Success =========="
echo "Executables location: ${BUILD_DIR}"
echo "  - scatter_update"
echo "  - scatter_update_kernel.so"
echo "  - scripts/"
echo "  - run_multi_data.sh"
echo ""
echo "To run: cd ${BUILD_DIR} && bash run_multi_data.sh"

### **6.3 运行脚本**

In [ ]:
%%writefile Sources/03.01/scatter_update/run_multi_data.sh
#!/bin/bash

BUILD_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"

if [ -n "${ASCEND_HOME_PATH}" ]; then
    ASCEND_DIR="${ASCEND_HOME_PATH}"
elif [ -n "${ASCEND_DIR}" ]; then
    ASCEND_DIR="${ASCEND_DIR}"
else
    echo "ERROR: ASCEND_HOME_PATH or ASCEND_DIR environment variable not set"
    echo "Please set ASCEND_HOME_PATH or ASCEND_DIR to your Ascend installation path"
    echo "Example: source /path/to/Ascend/cann/set_env.sh"
    exit 1
fi

export LD_LIBRARY_PATH="${BUILD_DIR}:${ASCEND_DIR}/lib64:${LD_LIBRARY_PATH}"

EXECUTABLE="${BUILD_DIR}/scatter_update"
SCRIPTS_DIR="${BUILD_DIR}/scripts"

if [ ! -f "$EXECUTABLE" ]; then
    echo "ERROR: Executable not found: $EXECUTABLE"
    echo "Please build the project first: bash build.sh"
    exit 1
fi

run_test_case() {
    local case_name=$1
    local m=$2
    local n=$3
    local k=$4
    
    local work_dir="${BUILD_DIR}/work_${case_name}"
    
    echo ""
    echo "=========================================="
    echo "Test Case: ${case_name}"
    echo "Parameters: m=${m}, n=${n}, k=${k}"
    echo "=========================================="
    
    mkdir -p "${work_dir}/input" "${work_dir}/output"
    
    cd "${work_dir}"
    python3 "${SCRIPTS_DIR}/gen_data.py" ${m} ${k} ${n}
    
    if [ $? -ne 0 ]; then
        echo "ERROR: [${case_name}] Data generation failed"
        return 1
    fi
    
    echo "Executing ScatterUpdate with pipeline optimization..."
    echo "ScatterUpdate output size: M*N = $(m * n) elements"
    
    ${EXECUTABLE} \
        --m ${m} \
        --n ${n} \
        --k ${k} \
        --output_dir "${work_dir}"
    
    if [ $? -ne 0 ]; then
        echo "ERROR: [${case_name}] Execution failed"
        return 1
    fi
    
    echo "Verifying results..."
    python3 "${SCRIPTS_DIR}/verify_result.py" ${RANK_SIZE} ${m} ${k} ${n}
    
    if [ $? -ne 0 ]; then
        echo "ERROR: [${case_name}] Verification failed"
        return 1
    fi
    
    echo "[${case_name}] PASSED"
    return 0
}

TEST_CASES=(
    "base_1024:1024:1024:1024"
    "large_2048:1000:2048:2048"
    "large_4096:3000:4096:4096"
)

TOTAL_PASSED=0
TOTAL_FAILED=0

for case in "${TEST_CASES[@]}"; do
    IFS=':' read -r name m n k <<< "$case"
    
    if run_test_case "$name" "$m" "$n" "$k"; then
        TOTAL_PASSED=$((TOTAL_PASSED + 1))
    else
        TOTAL_FAILED=$((TOTAL_FAILED + 1))
    fi
done

echo ""
echo "=========================================="
echo "Test Summary"
echo "=========================================="
echo "Total:  ${TOTAL_PASSED} passed, ${TOTAL_FAILED} failed"

if [ ${TOTAL_FAILED} -eq 0 ]; then
    echo "All test cases PASSED!"
    exit 0
else
    echo "Some test cases FAILED!"
    exit 1
fi

运行脚本特点：
- 内置 3 组测试 Case（1024/2048/4096）
- 全自动流程：生成数据 → 执行算子 → 精度验证

---
## **7. 测试与验证**

### **7.1 测试数据生成**

In [ ]:
%%writefile Sources/03.01/scatter_update/op_host/scripts/gen_data.py
#!/usr/bin/env python3
"""
Generate test data for ScatterUpdate operator.

ScatterUpdate operation:
    var[indices[i], :] = updates[i, :]
    - var shape: [VAR_ROW, COL] (float32)
    - indices shape: [ROW] (int32), values in range [0, VAR_ROW)
    - updates shape: [ROW, COL] (float32)

Usage:
    python3 gen_data.py <ROW> <COL> <VAR_ROW> [--seed <seed>] [--input-dir <dir>] [--output-dir <dir>]
"""

import argparse
import numpy as np
import os
import struct
import sys


def main():
    parser = argparse.ArgumentParser(
        description='Generate test data for ScatterUpdate operator')
    parser.add_argument('ROW', type=int, help='number of update rows')
    parser.add_argument('COL', type=int, help='number of columns')
    parser.add_argument('VAR_ROW', type=int, help='number of var rows')
    parser.add_argument('--seed', type=int, default=42,
                        help='Random seed (default: 42)')
    parser.add_argument('--input-dir', type=str, default='input',
                        help='Directory for input files (default: input/)')
    parser.add_argument('--output-dir', type=str, default='output',
                        help='Directory for output files (default: output/)')
    args = parser.parse_args()

    ROW, COL, VAR_ROW = args.ROW, args.COL, args.VAR_ROW
    print(f"========== ScatterUpdate Data Generation ==========")
    print(f"ROW (num update rows): {ROW}")
    print(f"COL (num columns):      {COL}")
    print(f"VAR_ROW (num var rows):     {VAR_ROW}")
    print(f"Random seed:          {args.seed}")

    rng = np.random.RandomState(args.seed)

    # Create input/output directories
    os.makedirs(args.input_dir, exist_ok=True)
    os.makedirs(args.output_dir, exist_ok=True)

    # ---- var: shape [VAR_ROW, COL], float32 ----
    var = rng.randn(VAR_ROW, COL).astype(np.float32)
    var.tofile(os.path.join(args.input_dir, 'var.bin'))
    
    # ---- indices: shape [ROW], int32, values in [0, VAR_ROW) with possible repeats ----
    indices = rng.randint(0, VAR_ROW, size=(ROW,)).astype(np.int32)
    indices.tofile(os.path.join(args.input_dir, 'indices.bin'))

    # ---- updates: shape [ROW, COL], float32 ----
    updates = rng.randn(ROW, COL).astype(np.float32)
    updates.tofile(os.path.join(args.input_dir, 'updates.bin'))

    # ---- Also save the inputs as .npy for later verification ----
    np.save(os.path.join(args.input_dir, 'var.npy'), var)
    np.save(os.path.join(args.input_dir, 'indices.npy'), indices)
    np.save(os.path.join(args.input_dir, 'updates.npy'), updates)

    # ---- Compute expected output for verification ----
    var_expected = var.copy()
    for i in range(ROW):
        var_expected[indices[i], :] = updates[i, :]
    var_expected.tofile(os.path.join(args.output_dir, 'var_expected.bin'))
    np.save(os.path.join(args.output_dir, 'var_expected.npy'), var_expected)

    # ---- Save metadata ----
    metadata_file = os.path.join(args.input_dir, 'metadata.txt')
    with open(metadata_file, 'w') as f:
        f.write(f"ROW={ROW}\n")
        f.write(f"COL={COL}\n")
        f.write(f"VAR_ROW={VAR_ROW}\n")
        f.write(f"seed={args.seed}\n")
        f.write(f"var_shape=({VAR_ROW},{COL})\n")
        f.write(f"indices_shape=({ROW},)\n")
        f.write(f"updates_shape=({ROW},{COL})\n")
        f.write(f"indices_values={indices.tolist()}\n")
    print(f"  Generated: {metadata_file}")

    print(f"========== Data Generation Complete ==========")


if __name__ == '__main__':
    main()

**数据生成流程：**
1. 输入：M, N, K, 表示输入参数的shape维度，var的shape为[M, N], indices的shape为[K], updates的shape为[K, N], var和indices为float类型，updates为int32类型，输入均支持随机和全1模式
3. CPU  参考输出：`VAR.float()` → shape[M,N]
4. Golden ScatterUpdate：输出只有索引位置的数据更新。


### **7.2 精度验证**

In [ ]:
%%writefile Sources/03.01/scatter_update/op_host/scripts/verify_result.py
#!/usr/bin/env python3
"""
Verify the output of ScatterUpdate kernel against a numpy reference.

ScatterUpdate operation:
    var[indices[i], :] = updates[i, :]
    - var shape: [VAR_ROW, COL] (float32)
    - indices shape: [ROW] (int32), values in range [0, VAR_ROW)
    - updates shape: [ROW, COL] (float32)

Usage:
    python3 verify_result.py <RANK_SIZE> <ROW> <COL> <VAR_ROW> [--input-dir <dir>]
                             [--output-dir <dir>]
"""

import argparse
import numpy as np
import os
import sys


def verify_result(var_initial, indices, updates, var_result,
                  rtol=1e-3, atol=1e-5):
    """
    Verify that var_result matches the expected scatter_update result.

    Args:
        var_initial: Original var array [VAR_ROW, COL]
        indices: Index array [ROW] (int32)
        updates: Updates array [ROW, COL]
        var_result: Kernel output var array [VAR_ROW, COL]
        rtol: Relative tolerance
        atol: Absolute tolerance

    Returns:
        (passed, max_diff, max_diff_idx, mismatch_count, var_expected, diff)
    """
    # Compute expected result using numpy reference
    var_expected = var_initial.copy()
    ROW = indices.shape[0]
    for i in range(ROW):
        var_expected[indices[i], :] = updates[i, :]

    # Compute element-wise difference
    diff = np.abs(var_result.astype(np.float64) -
                   var_expected.astype(np.float64))
    max_diff = np.max(diff)
    max_diff_idx = np.unravel_index(np.argmax(diff), diff.shape)

    # Check tolerance
    tolerance = atol + rtol * np.abs(var_expected.astype(np.float64))
    mismatch = diff > tolerance
    mismatch_count = np.sum(mismatch)

    passed = (mismatch_count == 0)

    return passed, max_diff, max_diff_idx, mismatch_count, var_expected, diff


def main():
    parser = argparse.ArgumentParser(
        description='Verify ScatterUpdate kernel output')
    parser.add_argument('rank_size', type=int, nargs='?', default=1,
                        help='Rank size (for compatibility, default: 1)')
    parser.add_argument('ROW', type=int, help='Number of update rows')
    parser.add_argument('COL', type=int, help='Number of columns')
    parser.add_argument('VAR_ROW', type=int, help='Number of var rows')
    parser.add_argument('--input-dir', type=str, default='input',
                        help='Directory for input files (default: input/)')
    parser.add_argument('--output-dir', type=str, default='output',
                        help='Directory for output files (default: output/)')
    parser.add_argument('--rtol', type=float, default=1e-3,
                        help='Relative tolerance (default: 1e-3)')
    parser.add_argument('--atol', type=float, default=1e-5,
                        help='Absolute tolerance (default: 1e-5)')
    parser.add_argument('--print-mismatches', type=int, default=5,
                        help='Number of mismatches to print (default: 5)')
    args = parser.parse_args()

    rank_size = args.rank_size
    ROW, COL, VAR_ROW = args.ROW, args.COL, args.VAR_ROW

    print(f"========== ScatterUpdate Verification ==========")
    print(f"ROW (num update rows): {ROW}")
    print(f"COL (num columns):     {COL}")
    print(f"VAR_ROW (num var rows):  {VAR_ROW}")
    print(f"Tolerance: rtol={args.rtol}, atol={args.atol}")

    # ---- Load input data ----
    # Try loading from .npy files first, then fall back to .bin files
    if os.path.exists(os.path.join(args.input_dir, 'var.npy')):
        var_init = np.load(os.path.join(args.input_dir, 'var.npy'))
    else:
        var_init = np.fromfile(os.path.join(args.input_dir, 'var.bin'),
                               dtype=np.float32).reshape(VAR_ROW, COL)

    if os.path.exists(os.path.join(args.input_dir, 'indices.npy')):
        indices = np.load(os.path.join(args.input_dir, 'indices.npy'))
    else:
        indices = np.fromfile(os.path.join(args.input_dir, 'indices.bin'),
                              dtype=np.int32).reshape(ROW)

    if os.path.exists(os.path.join(args.input_dir, 'updates.npy')):
        updates = np.load(os.path.join(args.input_dir, 'updates.npy'))
    else:
        updates = np.fromfile(os.path.join(args.input_dir, 'updates.bin'),
                              dtype=np.float32).reshape(ROW, COL)

    # ---- Load kernel output ----
    result_path = os.path.join(args.output_dir, 'var_updated.bin')
    if not os.path.exists(result_path):
        print(f"[ERROR] Output file not found: {result_path}")
        print("The kernel may not have run successfully.")
        sys.exit(1)

    var_result = np.fromfile(result_path, dtype=np.float32).reshape(VAR_ROW, COL)

    # ---- Verify ----
    passed, max_diff, max_diff_idx, mismatch_count, var_expected, diff = \
        verify_result(var_init, indices, updates, var_result,
                      rtol=args.rtol, atol=args.atol)

    # ---- Print results ----
    print(f"\n---- Verification Results ----")
    print(f"Max absolute difference: {max_diff:.6e}")
    print(f"Max diff location:       [{max_diff_idx[0]}, {max_diff_idx[1]}]")
    if max_diff_idx[0] < VAR_ROW and max_diff_idx[1] < COL:
        expected_val = var_expected[max_diff_idx[0], max_diff_idx[1]]
        result_val = var_result[max_diff_idx[0], max_diff_idx[1]]
        print(f"  Expected: {expected_val:.6e}")
        print(f"  Got:      {result_val:.6e}")
    total_elements = VAR_ROW * COL
    print(f"Mismatch count:          {mismatch_count} / {total_elements}")
    print(f"Mismatch ratio:          {mismatch_count / total_elements * 100:.4f}%")

    # Print first few mismatches
    if mismatch_count > 0 and args.print_mismatches > 0:
        print(f"\nFirst {min(args.print_mismatches, mismatch_count)} mismatches:")
        mismatch_indices = np.where(diff > args.atol + args.rtol *
                                    np.abs(var_expected.astype(np.float64)))
        count = 0
        for idx in zip(mismatch_indices[0], mismatch_indices[1]):
            if count >= args.print_mismatches:
                break
            print(f"  [{idx[0]}, {idx[1]}] expected={var_expected[idx]:.6e} "
                  f"got={var_result[idx]:.6e} diff={diff[idx]:.6e}")
            count += 1

    # ---- Summary ----
    print(f"\n=========================================")
    if passed:
        print(f"[PASSED] All {total_elements} elements match within tolerance.")
        print(f"=========================================")
        sys.exit(0)
    else:
        print(f"[FAILED] {mismatch_count} elements exceed tolerance.")
        print(f"=========================================")
        sys.exit(1)


if __name__ == '__main__':
    main()

**精度标准：**
- `rtol = 1e-2`（相对误差 1%）
- `atol = 4e-2`（绝对误差 0.04）
- 逐元素判定：`|npu - golden| <= atol + rtol × |golden|`

### **7.3 编译运行命令**

In [ ]:
# 注意：需要在有 NPU 卡的环境中执行
# 步骤1: 编译
!cd Sources/03.01/scatter_update && bash build.sh
# 步骤2: 运行测试
!cd build && bash run_multi_data.sh

---
## **8. 开发流程总结**

回顾 ScatterUpdate算子Simt开发的完整流程：

| 阶段 | 关键任务 | 输出物 |
|------|----------|--------|
| 1. 需求分析 | 确定算子输入输出参数 | 原型定义 |
| 2. Tiling设计 | 多核切分策略（均匀切分） | Tiling结构体 |
| 3. Kernel实现 | kernel计算块 | op_kernel/*.h |
| 4. Host实现 | Tiling构造、Kernel Launch | op_host/*.asc |
| 5. 测试验证 | Golden数据生成、多Case测试 | gen_data.py, verify_result.py |

---
# **课后实践**

### **练习1：修改 Tiling 参数观察行为**
手动修改 `aivCoreNum` 的值（尝试 64、56），观察 lengthPerBlock 和 tailBlockLength 的计算结果变化。提示：关注 `CreateTilingData` 中的校验和计算逻辑。

### **练习2：理解非确定性计算**
当前 ScatterUpdate 实现是非确定性计算，当indices取值重复时，可能会存在多核竞争。如果要保证用例每次都稳定PASS，那么需要怎么修改 indices 取值？

### **练习3：扩展思考**
如果要实现 ScatterAdd/ScatterSub 等Scatter类索引算子，与当前的实现上有哪些不同？需要修改哪些模块？

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/03.01_answer.txt